# Customer Value Radar : Audit qualité des données

Audit du dataset Online Retail II avant construction du pipeline analytique.

**Source**
- Online Retail II — UCI Machine Learning Repository
- Licence : CC BY 4.0
- Période : 01/12/2009 au 09/12/2011
- Volume : 1 067 371 observations
- E-commerçant britannique sans magasin physique
- Source du dataset : https://archive.ics.uci.edu/dataset/502/online+retail+ii

**Définitions UCI du dataset**
- `Invoice` : identifiant de facture ; préfixe `C` = annulation (cancellation)
- `StockCode` : code produit
- `Description` : nom du produit
- `Quantity` : quantité
- `InvoiceDate` : date et heure de la transaction
- `Price` : prix unitaire en livres sterling £
- `Customer ID` : identifiant ID client
- `Country` : pays de résidence du client



## 1. Chargement et structure des données

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)


In [2]:
# Chemin vers le fichier 

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "online_retail_II.xlsx"

print("Fichier source trouvé :", DATA_PATH.exists())


Fichier source trouvé : True


In [3]:
# Vérification des feuilles Excel

excel_file = pd.ExcelFile(DATA_PATH)

print(excel_file.sheet_names)


['Year 2009-2010', 'Year 2010-2011']


In [4]:
# Chargement des deux feuilles

df_2009_2010 = pd.read_excel(
    DATA_PATH,
    sheet_name="Year 2009-2010"
)

df_2010_2011 = pd.read_excel(
    DATA_PATH,
    sheet_name="Year 2010-2011"
)

print("2009-2010 :", df_2009_2010.shape)
print("2010-2011 :", df_2010_2011.shape)


2009-2010 : (525461, 8)
2010-2011 : (541910, 8)


In [5]:
print(df_2009_2010.dtypes)
print(df_2010_2011.dtypes)

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object


In [6]:
# Standardisation des types

for df in [df_2009_2010, df_2010_2011]:
    df["Invoice"] = df["Invoice"].astype("string")
    df["StockCode"] = df["StockCode"].astype("string")
    df["Description"] = df["Description"].astype("string")
    df["Country"] = df["Country"].astype("string")

    df["Customer ID"] = pd.to_numeric(
        df["Customer ID"],
        errors="coerce"
    ).astype("Int64")

    df["Quantity"] = pd.to_numeric(
        df["Quantity"],
        errors="raise"
    ).astype("int64")

    df["Price"] = pd.to_numeric(
        df["Price"],
        errors="raise"
    ).astype("float64")

    df["InvoiceDate"] = pd.to_datetime(
        df["InvoiceDate"],
        errors="raise"
    )


In [7]:
# Structure des deux feuilles

print("Colonnes 2009-2010 :")
print(df_2009_2010.columns.tolist())

print("\nColonnes 2010-2011 :")
print(df_2010_2011.columns.tolist())

print("\nTypes :")
print(df_2009_2010.dtypes)


Colonnes 2009-2010 :
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Colonnes 2010-2011 :
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Types :
Invoice                string
StockCode              string
Description            string
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             Int64
Country                string
dtype: object


In [8]:
# Valeurs manquantes et périodes

print("2009-2010")
print(df_2009_2010.isna().sum())
print(
    "Période :",
    df_2009_2010["InvoiceDate"].min(),
    "→",
    df_2009_2010["InvoiceDate"].max()
)

print("\n2010-2011")
print(df_2010_2011.isna().sum())
print(
    "Période :",
    df_2010_2011["InvoiceDate"].min(),
    "→",
    df_2010_2011["InvoiceDate"].max()
)


2009-2010
Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64
Période : 2009-12-01 07:45:00 → 2010-12-09 20:01:00

2010-2011
Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64
Période : 2010-12-01 08:26:00 → 2011-12-09 12:50:00


### Conclusion

- Year 2009-2010 : 525 461 lignes
- Year 2010-2011 : 541 910 lignes
- Les deux feuilles possèdent les mêmes 8 variables métier.
- Description et Customer ID contiennent des valeurs manquantes.
- Les périodes se chevauchent : ce point doit être vérifié avant concaténation.


## 2. Chevauchement entre les deux feuilles

In [9]:
# Période commune 

overlap_2009_2010 = df_2009_2010[
    (df_2009_2010["InvoiceDate"] >= "2010-12-01")
    & (df_2009_2010["InvoiceDate"] < "2010-12-10")
].copy()

overlap_2010_2011 = df_2010_2011[
    (df_2010_2011["InvoiceDate"] >= "2010-12-01")
    & (df_2010_2011["InvoiceDate"] < "2010-12-10")
].copy()

print("Lignes 2009-2010 :", len(overlap_2009_2010))
print("Lignes 2010-2011 :", len(overlap_2010_2011))

print(
    "Factures 2009-2010 :",
    overlap_2009_2010["Invoice"].nunique()
)

print(
    "Factures 2010-2011 :",
    overlap_2010_2011["Invoice"].nunique()
)


Lignes 2009-2010 : 22523
Lignes 2010-2011 : 22523
Factures 2009-2010 : 1088
Factures 2010-2011 : 1088


In [10]:
# Factures communes

invoices_a = set(overlap_2009_2010["Invoice"].dropna().unique())
invoices_b = set(overlap_2010_2011["Invoice"].dropna().unique())

common_invoices = invoices_a.intersection(invoices_b)

print("Factures communes :", len(common_invoices))


Factures communes : 1088


In [11]:
# Comparaison ligne par ligne

business_columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country"
]

overlap_a = (
    overlap_2009_2010[business_columns]
    .sort_values(business_columns, na_position="first")
    .reset_index(drop=True)
)

overlap_b = (
    overlap_2010_2011[business_columns]
    .sort_values(business_columns, na_position="first")
    .reset_index(drop=True)
)

print("Même nombre de lignes :", len(overlap_a) == len(overlap_b))
print("Lignes identiques :", overlap_a.equals(overlap_b))


Même nombre de lignes : True
Lignes identiques : True


### Conclusion

- Période commune : 1er au 9 décembre 2010
- 22 523 lignes transactionnelles dans chaque feuille
- 1 088 factures concernées
- Les lignes sont identiques après comparaison.

**Décision**
- conserver cette période uniquement depuis Year 2010-2011
- éviter un double comptage des lignes transactionnelles.


In [12]:
# Consolidation

df_2009_2010["SourcePeriod"] = "2009-2010"
df_2010_2011["SourcePeriod"] = "2010-2011"

df_2009_2010_unique = df_2009_2010[
    df_2009_2010["InvoiceDate"] < "2010-12-01"
].copy()

df_consolidated = pd.concat(
    [
        df_2009_2010_unique,
        df_2010_2011
    ],
    ignore_index=True
)

df_consolidated["SourcePeriod"] = (
    df_consolidated["SourcePeriod"]
    .astype("string")
)

print("Lignes consolidées :", len(df_consolidated))
print("Colonnes :", len(df_consolidated.columns))
print(
    "Période :",
    df_consolidated["InvoiceDate"].min(),
    "→",
    df_consolidated["InvoiceDate"].max()
)


Lignes consolidées : 1044848
Colonnes : 9
Période : 2009-12-01 07:45:00 → 2011-12-09 12:50:00


### Dataset de référence

- 1 044 848 lignes
- 9 colonnes, dont SourcePeriod
- Tous les contrôles suivants utilisent df_consolidated


## 3. Analyse des quantités négatives

In [13]:
negative_quantity = df_consolidated[
    df_consolidated["Quantity"] < 0
].copy()

negative_c = negative_quantity[
    negative_quantity["Invoice"].str.startswith("C", na=False)
].copy()

negative_without_c = negative_quantity[
    ~negative_quantity["Invoice"].str.startswith("C", na=False)
].copy()

print("Lignes avec Quantity < 0 :", len(negative_quantity))
print("Factures concernées :", negative_quantity["Invoice"].nunique())

print("\nNégatifs sur facture C :", len(negative_c))
print(
    "Part :",
    round(len(negative_c) / len(negative_quantity) * 100, 2),
    "%"
)

print("\nNégatifs sans facture C :", len(negative_without_c))
print(
    "Part :",
    round(len(negative_without_c) / len(negative_quantity) * 100, 2),
    "%"
)


Lignes avec Quantity < 0 : 22557
Factures concernées : 11684

Négatifs sur facture C : 19164
Part : 84.96 %

Négatifs sans facture C : 3393
Part : 15.04 %


In [14]:
# Vérification des factures C (cancelled) (annulées)

cancelled_lines = df_consolidated[
    df_consolidated["Invoice"].str.startswith("C", na=False)
].copy()

print("Lignes sur facture C :", len(cancelled_lines))
print("Quantity < 0 :", (cancelled_lines["Quantity"] < 0).sum())
print("Quantity = 0 :", (cancelled_lines["Quantity"] == 0).sum())
print("Quantity > 0 :", (cancelled_lines["Quantity"] > 0).sum())


Lignes sur facture C : 19165
Quantity < 0 : 19164
Quantity = 0 : 0
Quantity > 0 : 1


In [15]:
# Cas atypique : facture C avec quantité positive

c_positive_quantity = cancelled_lines[
    cancelled_lines["Quantity"] > 0
].copy()

display(
    c_positive_quantity[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country",
            "SourcePeriod"
        ]
    ]
)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourcePeriod
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,<NA>,United Kingdom,2009-2010


In [16]:
# Descriptions des quantités négatives hors C

display(
    negative_without_c["Description"]
    .value_counts(dropna=False)
    .head(20)
)


Description
<NA>                      2633
check                      121
damages                     83
?                           81
damaged                     78
missing                     27
sold as set on dotcom       20
Damaged                     17
smashed                      9
thrown away                  9
Unsaleable, destroyed.       9
dotcom                       8
damages?                     7
??                           7
crushed                      6
given away                   6
MIA                          5
Damages                      5
counted                      5
checked                      5
Name: count, dtype: int64[pyarrow]

### Conclusion sur les quantités négatives

- 22 557 lignes avec Quantity < 0
- 11 684 factures concernées
- 19 164 lignes (84,96 %) sur facture `C`
- 3 393 lignes (15,04 %) sans facture `C`
- une ligne sur facture `C` possède une quantité positive

UCI documente `C` comme une annulation (Cancelled)

La ligne concernée possède `StockCode = M` et `Description = Manual`. 
Elle est classée comme ligne manuelle et non comme vente produit. 
Cette classification repose sur les données observées

**Décision**
- conserver tous les mouvements négatifs ;
- identifier les factures `C` comme annulations ;
- identifier séparément les négatifs hors `C` ;
- signaler la ligne `C` à quantité positive comme ligne manuelle atypique ;



## 4. Analyse de la variable Price

In [17]:
negative_price = df_consolidated[
    df_consolidated["Price"] < 0
].copy()

zero_price = df_consolidated[
    df_consolidated["Price"] == 0
].copy()

positive_price = df_consolidated[
    df_consolidated["Price"] > 0
].copy()

print("Prix négatifs :", len(negative_price))
print("Prix nuls :", len(zero_price))
print("Prix positifs :", len(positive_price))


Prix négatifs : 5
Prix nuls : 6024
Prix positifs : 1038819


In [18]:
# Lignes avec prix négatif

display(
    negative_price[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "Customer ID",
            "Country"
        ]
    ]
)


,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,-53594.36,<NA>,United Kingdom
276274,A516228,B,Adjust bad debt,1,-44031.79,<NA>,United Kingdom
403472,A528059,B,Adjust bad debt,1,-38925.87,<NA>,United Kingdom
802921,A563186,B,Adjust bad debt,1,-11062.06,<NA>,United Kingdom
802922,A563187,B,Adjust bad debt,1,-11062.06,<NA>,United Kingdom


In [19]:
# Distribution des prix positifs

display(
    positive_price["Price"]
    .quantile([0.50, 0.90, 0.95, 0.99, 0.999])
)

print("Price > £200 :", (df_consolidated["Price"] > 200).sum())
print("Price > £1 000 :", (df_consolidated["Price"] > 1000).sum())
print("Price > £10 000 :", (df_consolidated["Price"] > 10000).sum())


0.500      2.1000
0.900      7.9500
0.950      9.9500
0.990     18.0000
0.999    216.5137
Name: Price, dtype: float64

Price > £200 : 1124
Price > £1 000 : 239
Price > £10 000 : 23


In [20]:
# Lignes avec Price > £1 000

high_price = df_consolidated[
    df_consolidated["Price"] > 1000
].copy()

print("Descriptions :")
display(
    high_price["Description"]
    .value_counts(dropna=False)
    .head(20)
)

print("StockCode :")
display(
    high_price["StockCode"]
    .value_counts(dropna=False)
    .head(20)
)


Descriptions :


Description
Manual                                 166
AMAZON FEE                              33
DOTCOM POSTAGE                          29
Bank Charges                             2
Discount                                 2
POSTAGE                                  2
CRUK Commission                          2
Adjustment by john on 26/01/2010 17      1
FLAG OF ST GEORGE CAR FLAG               1
Adjust bad debt                          1
Name: count, dtype: int64[pyarrow]

StockCode :


StockCode
M               166
AMAZONFEE        33
DOT              29
BANK CHARGES      2
D                 2
POST              2
CRUK              2
ADJUST            1
84016             1
B                 1
Name: count, dtype: int64[pyarrow]

In [21]:
# Contrôle d'un prix atypique

flag_product = df_consolidated[
    df_consolidated["Description"]
    .str.strip()
    .eq("FLAG OF ST GEORGE CAR FLAG")
].copy()

print("Nombre d'observations :", len(flag_product))

display(flag_product["Price"].describe())

display(
    flag_product["Price"]
    .value_counts()
    .sort_index()
)


Nombre d'observations : 73


count      73.00000
mean       45.55411
std       177.99440
min         0.00000
25%         0.42000
50%         0.42000
75%         0.42000
max      1157.15000
Name: Price, dtype: float64

Price
0.00        6
0.35        1
0.42       51
0.81        1
0.83        2
25.52       2
25.53       3
93.61       2
272.27      1
280.75      1
408.40      1
867.79      1
1157.15     1
Name: count, dtype: int64

### Conclusion sur les prix

**Prix négatifs**
- 5 lignes
- StockCode = B
- Description = Adjust bad debt
- UCI ne documente pas le code `B`

**Prix nuls**
- 6 024 lignes
- leur contribution à Quantity × Price est nulle ;
- leur nature métier n'est pas déduite du seul prix.

**Prix élevés**
- distribution fortement asymétrique 
- plusieurs libellés suggèrent des frais, commissions ou ajustements 
- `FLAG OF ST GEORGE CAR FLAG` montre qu'un seuil global ne suffit pas.

**Décision**
- aucune suppression sur un seuil de prix ;
- conserver les valeurs atypiques ;
- classifier les lignes selon leur contexte.


## 5. Analyse des Customer ID manquants

In [22]:
missing_customer = df_consolidated[
    df_consolidated["Customer ID"].isna()
].copy()

print("Lignes totales :", len(df_consolidated))
print("Customer ID manquants :", len(missing_customer))
print(
    "Part du dataset :",
    round(len(missing_customer) / len(df_consolidated) * 100, 2),
    "%"
)

print(
    "Factures concernées :",
    missing_customer["Invoice"].nunique()
)

print(
    "Pays concernés :",
    missing_customer["Country"].nunique()
)


Lignes totales : 1044848
Customer ID manquants : 235287
Part du dataset : 22.52 %
Factures concernées : 8752
Pays concernés : 15


In [23]:
# Lignes sans Customer ID avec quantite et prix positifs

positive_missing_customer = missing_customer[
    (missing_customer["Quantity"] > 0)
    & (missing_customer["Price"] > 0)
]

print(
    "Quantity > 0 et Price > 0 :",
    len(positive_missing_customer)
)


Quantity > 0 et Price > 0 : 228610


In [24]:
# Une facture mélange-t-elle des lignes identifiées et non identifiées ?

invoices_without_customer = set(
    missing_customer["Invoice"].dropna().unique()
)

invoices_with_customer = set(
    df_consolidated.loc[
        df_consolidated["Customer ID"].notna(),
        "Invoice"
    ].dropna().unique()
)

mixed_customer_invoices = (
    invoices_without_customer
    .intersection(invoices_with_customer)
)

print(
    "Factures avec lignes identifiées ET non identifiées :",
    len(mixed_customer_invoices)
)


Factures avec lignes identifiées ET non identifiées : 0


In [25]:
# Taux de Customer ID manquant par pays

missing_by_country = (
    missing_customer["Country"]
    .value_counts()
    .rename("missing_customer_rows")
    .to_frame()
)

all_by_country = (
    df_consolidated["Country"]
    .value_counts()
    .rename("total_rows")
    .to_frame()
)

missing_by_country = missing_by_country.join(all_by_country)

missing_by_country["missing_rate_pct"] = (
    missing_by_country["missing_customer_rows"]
    / missing_by_country["total_rows"]
    * 100
)

display(
    missing_by_country
    .sort_values("missing_customer_rows", ascending=False)
)


,missing_customer_rows,total_rows,missing_rate_pct
Country,,,
United Kingdom,232320,959983,24.200429
EIRE,1660,17689,9.384363
Hong Kong,364,364,100.0
Unspecified,232,756,30.687831
France,128,14059,0.910449
Switzerland,125,3183,3.927113
Portugal,116,2540,4.566929
United Arab Emirates,114,500,22.8
Bahrain,67,126,53.174603


### Conclusion sur les Customer ID

- 235 287 lignes sans `Customer ID`
- 22,52 % du dataset consolidé
- aucune facture ne mélange lignes identifiées et non identifiées
- le phénomène est très majoritairement concentré au Royaume-Uni

**Décision**
- aucune imputation artificielle ;
- conserver pour les analyses commerciales 
- exclure des analyses clients individuelles 



## 6. Analyse des descriptions manquantes

In [26]:
missing_description = df_consolidated[
    df_consolidated["Description"].isna()
].copy()

print("Descriptions manquantes :", len(missing_description))
print(
    "Part du dataset :",
    round(len(missing_description) / len(df_consolidated) * 100, 2),
    "%"
)

print("Quantity < 0 :", (missing_description["Quantity"] < 0).sum())
print("Quantity > 0 :", (missing_description["Quantity"] > 0).sum())
print("Price = 0 :", (missing_description["Price"] == 0).sum())
print("Price > 0 :", (missing_description["Price"] > 0).sum())

print(
    "Customer ID manquant :",
    missing_description["Customer ID"].isna().sum()
)


Descriptions manquantes : 4275
Part du dataset : 0.41 %
Quantity < 0 : 2633
Quantity > 0 : 1642
Price = 0 : 4275
Price > 0 : 0
Customer ID manquant : 4275


In [27]:
# StockCode concernés

display(
    missing_description["StockCode"]
    .value_counts(dropna=False)
    .head(30)
)


StockCode
84990     12
79321     11
22950     10
35965     10
23084     10
22139      9
22087      9
22084      9
71477      8
35970      7
21768      7
22528      7
POST       7
84795D     7
22451      7
37446      6
21478      6
21169      6
72803B     6
21340      6
84845C     6
20747      6
90084      6
84977      6
84467      6
37461      6
22467      6
20734      6
22689      6
21784      6
Name: count, dtype: int64[pyarrow]

### Conclusion sur les descriptions manquantes

- 4 275 lignes, soit 0,41 %
- 2 633 avec Quantity < 0
- 1 642 avec Quantity > 0
- toutes ont Price = 0
- toutes sont sans Customer ID

**Décision**
- conserver les observations 
- aucune imputation de Description
- exclure des analyses nécessitant un libellé produit 
- leur contribution à Quantity × Price est nulle.


## 7. Observations strictement identiques

In [28]:
# Doublons exacts après consolidation

exact_duplicates = df_consolidated.duplicated().sum()

exact_duplicate_rows = df_consolidated[
    df_consolidated.duplicated(keep=False)
].copy()

print("Doublons exacts excédentaires :", exact_duplicates)

print(
    "Lignes appartenant à un groupe de répétitions :",
    len(exact_duplicate_rows)
)

print(
    "Factures concernées :",
    exact_duplicate_rows["Invoice"].nunique()
)

print(
    "Part des doublons exacts :",
    round(exact_duplicates / len(df_consolidated) * 100, 2),
    "%"
)


Doublons exacts excédentaires : 11812
Lignes appartenant à un groupe de répétitions : 22813
Factures concernées : 4387
Part des doublons exacts : 1.13 %


In [29]:
# Exemple concret de lignes identiques

display(
    exact_duplicate_rows[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country"
        ]
    ]
    .sort_values(
        ["Invoice", "StockCode", "InvoiceDate"]
    )
    .head(30)
)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329,United Kingdom


In [30]:
# Taille des groupes de répétitions

duplicate_group_sizes = (
    df_consolidated
    .groupby(
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country"
        ],
        dropna=False
    )
    .size()
)

duplicate_group_sizes = duplicate_group_sizes[
    duplicate_group_sizes > 1
]

display(
    duplicate_group_sizes
    .value_counts()
    .sort_index()
)


2     10352
3       546
4        79
5        11
6        10
8         1
12        1
20        1
Name: count, dtype: int64

In [31]:
# Simulation uniquement : impact potentiel d'une déduplication
# Aucune suppression n'est appliquée à df_consolidated.

df_without_exact_duplicates = (
    df_consolidated
    .drop_duplicates()
)

print("Avant :", len(df_consolidated))
print("Après simulation :", len(df_without_exact_duplicates))

print(
    "Lignes qui seraient retirées :",
    len(df_consolidated) - len(df_without_exact_duplicates)
)

print(
    "Réduction du volume :",
    round(
        (
            len(df_consolidated)
            - len(df_without_exact_duplicates)
        )
        / len(df_consolidated)
        * 100,
        2
    ),
    "%"
)


Avant : 1044848
Après simulation : 1033036
Lignes qui seraient retirées : 11812
Réduction du volume : 1.13 %


In [32]:
# Sensibilité sur la valeur transactionnelle brute
# Ce calcul n'est pas le chiffre d'affaires final.

value_all = (
    df_consolidated["Quantity"]
    * df_consolidated["Price"]
).sum()

value_deduplicated = (
    df_without_exact_duplicates["Quantity"]
    * df_without_exact_duplicates["Price"]
).sum()

difference = value_all - value_deduplicated

print("Valeur brute avec toutes les lignes :", round(value_all, 2))
print("Valeur brute après simulation :", round(value_deduplicated, 2))
print("Écart :", round(difference, 2))

print(
    "Écart relatif :",
    round(difference / value_all * 100, 4),
    "%"
)


Valeur brute avec toutes les lignes : 18909762.12
Valeur brute après simulation : 18855533.7
Écart : 54228.42
Écart relatif : 0.2868 %


### Conclusion sur les observations identiques

Après consolidation :

- 11 812 lignes seraient retirées par `drop_duplicates()`
- 22 813 lignes appartiennent à un groupe de répétitions
- 4 387 factures concernées
- réduction potentielle du volume : 1,13 %
- écart sur la valeur brute `Quantity × Price` : 0,2868 %

Le dataset ne fournit pas d'identifiant unique de ligne de facture.

**Décision**
- aucune suppression automatique 
- conserver les observations dans la couche auditable 
- signaler les répétitions 
- définir le grain analytique avant toute déduplication


## 8. Contrôles avant modélisation

Controles pour fixer les futures règles Silver.

Ils portent sur :
- les types de factures ;
- les StockCode spéciaux ;
- le grain d'une facture ;
- la stabilité du pays client ;
- la stabilité des descriptions produit ;
- les quantités extrêmes.


### Préfixes de facture

In [33]:
invoice_prefix = (
    df_consolidated["Invoice"]
    .str.extract(r"^([A-Za-z]+)", expand=False)
    .fillna("NUMERIC")
)

display(
    invoice_prefix.value_counts()
)


Invoice
NUMERIC    1025677
C            19165
A                6
Name: count, dtype: int64[pyarrow]

In [34]:
# Factures avec préfixe A

a_invoices = df_consolidated[
    df_consolidated["Invoice"]
    .str.startswith("A", na=False)
].copy()

print("Lignes A :", len(a_invoices))
print("Factures A distinctes :", a_invoices["Invoice"].nunique())

display(
    a_invoices[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country"
        ]
    ].sort_values("InvoiceDate")
)


Lignes A : 6
Factures A distinctes : 6


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,<NA>,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,<NA>,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,<NA>,United Kingdom
802920,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,<NA>,United Kingdom
802921,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,<NA>,United Kingdom
802922,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,<NA>,United Kingdom


**Observation**
- trois formes de facture sont observées : numérique, `C` et `A` ;
- UCI documente uniquement `C` comme annulation 
- les 6 lignes `A` portent toutes `StockCode = B` et `Description = Adjust bad debt` 
- une écriture positive est suivie de deux écritures négatives de même montant à une minute d'intervalle.

**Décision**
- ne pas donner de signification générale au préfixe `A` 
- classer ces 6 lignes comme ajustements financiers à partir de leur contenu observé.


### `StockCode` spéciaux

In [35]:
special_stockcodes = df_consolidated[
    ~df_consolidated["StockCode"]
    .str.match(r"^\d", na=False)
].copy()

print("Lignes concernées :", len(special_stockcodes))
print(
    "StockCode distincts :",
    special_stockcodes["StockCode"].nunique()
)

display(
    special_stockcodes["StockCode"]
    .value_counts()
    .head(50)
)


Lignes concernées : 5992
StockCode distincts : 62


StockCode
POST            2086
DOT             1425
M               1398
C2               277
D                173
S                102
BANK CHARGES     100
ADJUST            67
AMAZONFEE         36
DCGS0058          31
gift_0001_20      29
gift_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
gift_0001_10      16
CRUK              16
TEST001           15
DCGS0076          14
DCGS0003          14
gift_0001_50       8
gift_0001_40       7
DCGS0069           6
B                  6
DCGS0004           5
m                  5
gift_0001_80       4
DCGS0072           4
DCGS0066N          4
DCGS0068           3
gift_0001_70       3
DCGS0070           3
ADJUST2            3
SP1002             3
TEST002            2
gift_0001_60       2
gift_0001_90       2
DCGS0062           2
DCGS0037           2
DCGS0044           1
DCGS0006           1
DCGS0016           1
DCGS0027           1
DCGS0036           1
DCGS0039           1
DCGS0060           1
DCGS0056           1
DCG

In [36]:
# Résumé des principaux StockCode spéciaux

special_code_summary = (
    special_stockcodes
    .groupby("StockCode", dropna=False)
    .agg(
        rows=("StockCode", "size"),
        descriptions=("Description", "nunique"),
        min_price=("Price", "min"),
        max_price=("Price", "max"),
        total_quantity=("Quantity", "sum")
    )
    .sort_values("rows", ascending=False)
)

display(
    special_code_summary.head(50)
)


,rows,descriptions,min_price,max_price,total_quantity
StockCode,,,,,
POST,2086,1,0.000,8142.75,10011
DOT,1425,1,0.000,4505.17,2917
M,1398,1,0.000,38970.00,4494
C2,277,1,0.000,150.00,711
D,173,1,0.010,1867.86,-2868
S,102,1,2.800,605.18,-96
BANK CHARGES,100,2,0.001,18910.69,-42
ADJUST,67,3,4.570,5117.03,5
AMAZONFEE,36,1,1.000,17836.46,-30


In [37]:
# Inventaire détaillé des StockCode spéciaux

special_stockcodes_analysis = special_stockcodes.copy()

special_stockcodes_analysis["is_cancellation"] = (
    special_stockcodes_analysis["Invoice"]
    .str.startswith("C", na=False)
)

special_stockcodes_analysis["is_negative_quantity"] = (
    special_stockcodes_analysis["Quantity"] < 0
)

special_stockcodes_analysis["is_zero_price"] = (
    special_stockcodes_analysis["Price"] == 0
)

special_stockcode_inventory = (
    special_stockcodes_analysis
    .groupby("StockCode")
    .agg(
        rows=("StockCode", "size"),
        descriptions=("Description", "nunique"),
        cancellations=("is_cancellation", "sum"),
        negative_quantity=("is_negative_quantity", "sum"),
        zero_price=("is_zero_price", "sum"),
        min_price=("Price", "min"),
        max_price=("Price", "max"),
        total_quantity=("Quantity", "sum")
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

display(special_stockcode_inventory)

,StockCode,rows,descriptions,cancellations,negative_quantity,zero_price,min_price,max_price,total_quantity
47,POST,2086,1,228,228,7,0.00,8142.75,10011
43,DOT,1425,1,3,3,7,0.00,4505.17,2917
45,M,1398,1,535,534,7,0.00,38970.00,4494
5,C2,277,1,7,7,3,0.00,150.00,711
8,D,173,1,168,168,0,0.01,1867.86,-2868
...,...,...,...,...,...,...,...,...,...
33,DCGS0071,1,0,0,1,1,0.00,0.00,-2
29,DCGS0067,1,1,0,1,1,0.00,0.00,-11
28,DCGS0066P,1,0,0,1,1,0.00,0.00,-3
25,DCGS0060,1,0,0,1,1,0.00,0.00,-3


In [38]:
# Descriptions observées pour chaque StockCode spécial

for stockcode in special_stockcode_inventory["StockCode"]:
    descriptions = (
        special_stockcodes_analysis.loc[
            special_stockcodes_analysis["StockCode"] == stockcode,
            "Description"
        ]
        .value_counts(dropna=False)
        .head(10)
    )

    print("\nStockCode :", stockcode)
    display(descriptions)


StockCode : POST


Description
POSTAGE    2079
<NA>          7
Name: count, dtype: int64[pyarrow]


StockCode : DOT


Description
DOTCOM POSTAGE    1423
<NA>                 2
Name: count, dtype: int64[pyarrow]


StockCode : M


...

Description
<NA>    1
Name: count, dtype: int64[pyarrow]


StockCode : DCGS0060


Description
<NA>    1
Name: count, dtype: int64[pyarrow]


StockCode : DCGS0053


Description
<NA>    1
Name: count, dtype: int64[pyarrow]

In [39]:
# Description principale de chaque StockCode spécial

description_counts = (
    special_stockcodes_analysis
    .groupby(
        ["StockCode", "Description"],
        dropna=False
    )
    .size()
    .reset_index(name="description_rows")
)

main_description = (
    description_counts
    .sort_values(
        ["StockCode", "description_rows"],
        ascending=[True, False]
    )
    .drop_duplicates("StockCode")
    [["StockCode", "Description", "description_rows"]]
)

special_stockcode_review = (
    special_stockcode_inventory
    .merge(
        main_description,
        on="StockCode",
        how="left"
    )
    .sort_values("rows", ascending=False)
)

display(special_stockcode_review)

,StockCode,rows,descriptions,cancellations,negative_quantity,zero_price,min_price,max_price,total_quantity,Description,description_rows
0,POST,2086,1,228,228,7,0.00,8142.75,10011,POSTAGE,2079
1,DOT,1425,1,3,3,7,0.00,4505.17,2917,DOTCOM POSTAGE,1423
2,M,1398,1,535,534,7,0.00,38970.00,4494,Manual,1398
3,C2,277,1,7,7,3,0.00,150.00,711,CARRIAGE,274
4,D,173,1,168,168,0,0.01,1867.86,-2868,Discount,173
...,...,...,...,...,...,...,...,...,...,...,...
42,DCGS0027,1,0,0,1,1,0.00,0.00,-1,<NA>,1
41,DCGS0039,1,0,0,1,1,0.00,0.00,-1,<NA>,1
40,DCGS0016,1,0,0,1,1,0.00,0.00,-1,<NA>,1
39,DCGS0006,1,0,0,1,1,0.00,0.00,-1,<NA>,1


In [40]:
# Première classification des StockCode spéciaux

shipping_codes = [
    "POST",
    "DOT",
    "C2"
]

discount_codes = [
    "D"
]

sample_codes = [
    "S"
]

fee_codes = [
    "BANK CHARGES",
    "AMAZONFEE",
    "CRUK"
]

adjustment_codes = [
    "ADJUST",
    "ADJUST2",
    "B"
]

manual_codes = [
    "M",
    "m"
]

test_codes = [
    "TEST001",
    "TEST002"
]

gift_voucher_codes = [
    "gift_0001_10",
    "gift_0001_20",
    "gift_0001_30",
    "gift_0001_40",
    "gift_0001_50",
    "gift_0001_60",
    "gift_0001_70",
    "gift_0001_80",
    "gift_0001_90"
]

product_codes = [
    "DCGS0058",
    "DCGSSGIRL",
    "DCGSSBOY",
    "PADS",
    "DCGS0003",
    "DCGS0076",
    "DCGS0069",
    "DCGS0004",
    "DCGS0072",
    "DCGS0066N",
    "DCGS0068",
    "DCGS0070",
    "SP1002",
    "DCGS0037",
    "DCGS0062",
    "DCGS0041",
    "DCGS0044",
    "DCGS0075"
]

In [41]:
# Attribution d'une catégorie

special_stockcode_review["category"] = "unknown"

special_stockcode_review.loc[
    special_stockcode_review["StockCode"].isin(shipping_codes),
    "category"
] = "shipping"

special_stockcode_review.loc[
    special_stockcode_review["StockCode"].isin(discount_codes),
    "category"
] = "discount"

special_stockcode_review.loc[
    special_stockcode_review["StockCode"].isin(sample_codes),
    "category"
] = "sample"

special_stockcode_review.loc[
    special_stockcode_review["StockCode"].isin(fee_codes),
    "category"
] = "fee"

special_stockcode_review.loc[
    special_stockcode_review["StockCode"].isin(adjustment_codes),
    "category"
] = "adjustment"

special_stockcode_review.loc[
    special_stockcode_review["StockCode"].isin(manual_codes),
    "category"
] = "manual"

special_stockcode_review.loc[
    special_stockcode_review["StockCode"].isin(test_codes),
    "category"
] = "test"

special_stockcode_review.loc[
    special_stockcode_review["StockCode"].isin(gift_voucher_codes),
    "category"
] = "gift_voucher"

special_stockcode_review.loc[
    special_stockcode_review["StockCode"].isin(product_codes),
    "category"
] = "product"

display(
    special_stockcode_review[
        [
            "StockCode",
            "Description",
            "rows",
            "category"
        ]
    ]
)

,StockCode,Description,rows,category
0,POST,POSTAGE,2086,shipping
1,DOT,DOTCOM POSTAGE,1425,shipping
2,M,Manual,1398,manual
3,C2,CARRIAGE,277,shipping
4,D,Discount,173,discount
...,...,...,...,...
42,DCGS0027,<NA>,1,unknown
41,DCGS0039,<NA>,1,unknown
40,DCGS0016,<NA>,1,unknown
39,DCGS0006,<NA>,1,unknown


In [42]:
# Contrôle de la classification

print("Nombre total de StockCode :", len(special_stockcode_review))

display(
    special_stockcode_review["category"]
    .value_counts()
)

print("\nStockCode encore non classés :")

display(
    special_stockcode_review.loc[
        special_stockcode_review["category"] == "unknown",
        [
            "StockCode",
            "Description",
            "rows"
        ]
    ]
)

Nombre total de StockCode : 62


category
unknown         20
product         18
gift_voucher     9
shipping         3
fee              3
adjustment       3
manual           2
test             2
discount         1
sample           1
Name: count, dtype: int64


StockCode encore non classés :


,StockCode,Description,rows
55,DCGS0074,<NA>,1
52,DCGSLBOY,<NA>,1
53,DCGS0056,<NA>,1
58,DCGS0067,ebay,1
56,DCGS0073,ebay,1
57,DCGS0071,<NA>,1
50,DCGS0055,<NA>,1
59,DCGS0066P,<NA>,1
60,DCGS0060,<NA>,1
51,DCGSLGIRL,<NA>,1


In [43]:
# Détail des StockCode encore non classés

unknown_codes = special_stockcode_review.loc[
    special_stockcode_review["category"] == "unknown",
    "StockCode"
]

unknown_rows = df_consolidated[
    df_consolidated["StockCode"].isin(unknown_codes)
].copy()

display(
    unknown_rows[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country"
        ]
    ].sort_values("StockCode")
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
302041,518785,C3,<NA>,-4999,2010-08-11 16:55:00,0.0,<NA>,United Kingdom
80337,496740,DCGS0006,<NA>,-1,2010-02-03 14:28:00,0.0,<NA>,United Kingdom
80338,496741,DCGS0016,<NA>,-1,2010-02-03 14:29:00,0.0,<NA>,United Kingdom
80339,496742,DCGS0027,<NA>,-1,2010-02-03 14:29:00,0.0,<NA>,United Kingdom
80342,496744,DCGS0036,<NA>,-1,2010-02-03 14:35:00,0.0,<NA>,United Kingdom
80361,496746,DCGS0039,<NA>,-1,2010-02-03 14:36:00,0.0,<NA>,United Kingdom
98950,498698,DCGS0053,<NA>,-2,2010-02-22 12:29:00,0.0,<NA>,United Kingdom
577775,542529,DCGS0055,<NA>,-1,2011-01-28 13:08:00,0.0,<NA>,United Kingdom
81143,496855,DCGS0056,<NA>,-4,2010-02-04 11:46:00,0.0,<NA>,United Kingdom
578233,542582,DCGS0057,<NA>,-6,2011-01-28 15:01:00,0.0,<NA>,United Kingdom


In [44]:
# Vue compacte des StockCode encore non classés

unknown_review = unknown_rows[
    [
        "StockCode",
        "Description",
        "Quantity",
        "Price",
        "Invoice",
        "InvoiceDate"
    ]
].sort_values("StockCode")

print(
    unknown_review.to_string(index=False)
)

StockCode Description  Quantity  Price Invoice         InvoiceDate
       C3        <NA>     -4999    0.0  518785 2010-08-11 16:55:00
 DCGS0006        <NA>        -1    0.0  496740 2010-02-03 14:28:00
 DCGS0016        <NA>        -1    0.0  496741 2010-02-03 14:29:00
 DCGS0027        <NA>        -1    0.0  496742 2010-02-03 14:29:00
 DCGS0036        <NA>        -1    0.0  496744 2010-02-03 14:35:00
 DCGS0039        <NA>        -1    0.0  496746 2010-02-03 14:36:00
 DCGS0053        <NA>        -2    0.0  498698 2010-02-22 12:29:00
 DCGS0055        <NA>        -1    0.0  542529 2011-01-28 13:08:00
 DCGS0056        <NA>        -4    0.0  496855 2010-02-04 11:46:00
 DCGS0057        <NA>        -6    0.0  542582 2011-01-28 15:01:00
 DCGS0059        <NA>        -8    0.0  496988 2010-02-05 11:21:00
 DCGS0060        <NA>        -3    0.0  496814 2010-02-04 09:46:00
DCGS0066P        <NA>        -3    0.0  561255 2011-07-26 11:52:00
 DCGS0067        ebay       -11    0.0  561254 2011-07-26 11:5

In [45]:
# Classification des StockCode encore non classés

stock_movement_codes = unknown_review["StockCode"].tolist()

special_stockcode_review.loc[
    special_stockcode_review["StockCode"].isin(stock_movement_codes),
    "category"
] = "stock_movement"

print(
    special_stockcode_review["category"]
    .value_counts()
)

print(
    "\nStockCode encore non classés :",
    (
        special_stockcode_review["category"]
        == "unknown"
    ).sum()
)

category
stock_movement    20
product           18
gift_voucher       9
shipping           3
fee                3
adjustment         3
manual             2
test               2
discount           1
sample             1
Name: count, dtype: int64

StockCode encore non classés : 0


In [ ]:
# Export de la classification des StockCode

REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"

REFERENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

stockcode_categories = (
    special_stockcode_review[
        ["StockCode", "category"]
    ]
    .rename(
        columns={
            "StockCode": "stock_code"
        }
    )
    .sort_values("stock_code")
    .reset_index(drop=True)
)

OUTPUT_FILE = (
    REFERENCE_DIR
    / "stockcode_categories.csv"
)

stockcode_categories.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Fichier créé :", OUTPUT_FILE.name)
print("Nombre de codes :", len(stockcode_categories))

display(stockcode_categories)

### Conclusion sur la classification des StockCode spéciaux

Les 62 `StockCode` ne commençant pas par un chiffre ont été analysés à partir de leurs descriptions, quantités, prix et contexte transactionnel.

Les catégories retenues sont :

- `shipping` : frais ou opérations de livraison ;
- `discount` : remises ;
- `sample` : échantillons ;
- `fee` : frais et commissions ;
- `adjustment` : ajustements financiers ;
- `manual` : lignes manuelles ;
- `test` : données de test ;
- `gift_voucher` : bons cadeaux ;
- `product` : codes dont les descriptions observées correspondent à des articles ;
- `stock_movement` : mouvements négatifs à prix nul ne correspondant pas à des ventes observées.

Les 20 codes initialement non classés apparaissent uniquement avec des quantités négatives et un prix nul. Ils sont donc classés comme `stock_movement` à partir de leur comportement observé dans ce dataset.

Cette classification est une règle analytique du projet et non une définition officielle fournie par UCI.

**Décision**
- conserver toutes les lignes dans la couche auditable ;
- ajouter une catégorie de ligne dans Silver ;
- exclure les catégories non pertinentes lorsque des KPI spécifiques aux ventes produit sont calculés ;
- conserver la possibilité d'analyser séparément frais, remises, ajustements et mouvements de stock.

###  Grain de la facture

In [46]:
invoice_grain = (
    df_consolidated
    .groupby("Invoice", dropna=False)
    .agg(
        invoice_dates=("InvoiceDate", "nunique"),
        customer_ids=("Customer ID", "nunique"),
        countries=("Country", "nunique")
    )
)

print("Maximum de dates par facture :", invoice_grain["invoice_dates"].max())
print("Maximum de clients par facture :", invoice_grain["customer_ids"].max())
print("Maximum de pays par facture :", invoice_grain["countries"].max())

print()
print(
    "Factures avec plusieurs dates :",
    (invoice_grain["invoice_dates"] > 1).sum()
)
print(
    "Factures avec plusieurs clients :",
    (invoice_grain["customer_ids"] > 1).sum()
)
print(
    "Factures avec plusieurs pays :",
    (invoice_grain["countries"] > 1).sum()
)


Maximum de dates par facture : 2
Maximum de clients par facture : 1
Maximum de pays par facture : 1

Factures avec plusieurs dates : 83
Factures avec plusieurs clients : 0
Factures avec plusieurs pays : 0


In [47]:
# Factures avec plusieurs horodatages

multi_date_invoice_ids = invoice_grain[
    invoice_grain["invoice_dates"] > 1
].index

multi_date_rows = df_consolidated[
    df_consolidated["Invoice"].isin(multi_date_invoice_ids)
].copy()

multi_date_summary = (
    multi_date_rows
    .groupby("Invoice")
    .agg(
        first_date=("InvoiceDate", "min"),
        last_date=("InvoiceDate", "max"),
        rows=("Invoice", "size")
    )
    .reset_index()
)

multi_date_summary["gap_minutes"] = (
    multi_date_summary["last_date"]
    - multi_date_summary["first_date"]
).dt.total_seconds() / 60

print("Factures multi-dates :", len(multi_date_summary))

display(
    multi_date_summary["gap_minutes"]
    .describe()
)

display(
    multi_date_summary
    .sort_values("gap_minutes", ascending=False)
    .head(30)
)


Factures multi-dates : 83


count    83.000000
mean      1.144578
std       0.938766
min       1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       9.000000
Name: gap_minutes, dtype: float64

,Invoice,first_date,last_date,rows,gap_minutes
32,529368,2010-10-28 10:07:00,2010-10-28 10:16:00,30,9.0
21,520878,2010-08-31 15:32:00,2010-08-31 15:36:00,116,4.0
25,521901,2010-09-09 12:35:00,2010-09-09 12:37:00,49,2.0
60,547690,2011-03-24 14:55:00,2011-03-24 14:56:00,23,1.0
59,546986,2011-03-18 12:55:00,2011-03-18 12:56:00,46,1.0
58,546388,2011-03-11 13:42:00,2011-03-11 13:43:00,90,1.0
57,545713,2011-03-07 10:11:00,2011-03-07 10:12:00,120,1.0
56,545460,2011-03-02 17:32:00,2011-03-02 17:33:00,80,1.0
55,544926,2011-02-24 17:50:00,2011-02-24 17:51:00,6,1.0
54,544667,2011-02-22 15:09:00,2011-02-22 15:10:00,34,1.0


In [48]:
# Même jour ou plusieurs jours ?

same_day = (
    multi_date_summary["first_date"].dt.date
    == multi_date_summary["last_date"].dt.date
)

print("Même jour :", same_day.sum())
print("Plusieurs jours :", (~same_day).sum())


Même jour : 83
Plusieurs jours : 0


**Observation**
- une facture possède au maximum **1 client** ;
- une facture possède au maximum **1 pays** ;
- **83 factures** possèdent deux horodatages ;
- les 83 restent sur une même journée ;
- l'écart médian est de **1 minute** ;
- l'écart maximal est de **9 minutes**.

**Décision**
- le grain client et pays est cohérent au niveau `Invoice` ;
- les écarts observés sont compatibles avec plusieurs horodatages de lignes appartenant à une même facture ;
- aucune réutilisation d'un identifiant de facture sur plusieurs jours n'est observée ;
- convention de modélisation : `order_datetime` = `MIN(InvoiceDate)` de la facture ;
- `fact_order_lines` conserve l'horodatage propre à chaque ligne.


### 8.4 Clients présents dans plusieurs pays

In [49]:
customer_country = (
    df_consolidated[
        df_consolidated["Customer ID"].notna()
    ]
    .groupby("Customer ID")
    .agg(
        countries=("Country", "nunique")
    )
)

multi_country_customers = customer_country[
    customer_country["countries"] > 1
]

print(
    "Clients présents dans plusieurs pays :",
    len(multi_country_customers)
)

display(
    multi_country_customers
    .sort_values("countries", ascending=False)
)


Clients présents dans plusieurs pays : 13


,countries
Customer ID,
12370,2
12394,2
12413,2
12417,2
12422,2
12423,2
12429,2
12431,2
12449,2


**Observation**
- **13 clients** apparaissent dans deux pays.

**Décision**
- `Country` ne doit pas être traité comme un attribut client parfaitement stable ;
- conserver le pays au niveau de la transaction ou de la commande.


###  `StockCode` et descriptions

In [50]:
stockcode_descriptions = (
    df_consolidated
    .groupby("StockCode", dropna=False)["Description"]
    .nunique()
    .sort_values(ascending=False)
)

display(
    stockcode_descriptions.head(30)
)


StockCode
20713     9
22423     7
23084     7
21181     7
22734     7
22719     6
21830     6
85175     6
47566B    6
22501     5
21843     5
72807A    5
85123A    5
22740     5
85172     5
23343     5
23131     5
22502     5
21621     4
72807B    4
37479P    4
23209     4
22236     4
23203     4
84997C    4
23196     4
22176     4
20685     4
22139     4
79000     4
Name: Description, dtype: int64

In [51]:
# Exemples pour les StockCode avec plusieurs descriptions

top_stockcodes = (
    stockcode_descriptions[
        stockcode_descriptions > 1
    ]
    .head(5)
    .index
)

for stockcode in top_stockcodes:
    print("\nStockCode :", stockcode)

    display(
        df_consolidated.loc[
            df_consolidated["StockCode"] == stockcode,
            "Description"
        ]
        .value_counts(dropna=False)
        .head(15)
    )



StockCode : 20713


Description
JUMBO BAG OWLS                  1346
<NA>                               5
missing                            1
wrongly marked. 23343 in box       1
wrongly coded-23343                1
found                              1
Found                              1
wrongly marked 23343               1
Marked as 23343                    1
wrongly coded 23343                1
Name: count, dtype: int64[pyarrow]


StockCode : 22423


Description
REGENCY CAKESTAND 3 TIER    4316
damaged                        3
smashed                        2
damages                        2
<NA>                           1
broken, uneven bottom          1
wonky bottom/broken            1
faulty                         1
Name: count, dtype: int64[pyarrow]


StockCode : 23084


Description
RABBIT NIGHT LIGHT                     1051
<NA>                                     10
temp adjustment                           1
allocate stock for dotcom orders ta       1
add stock to allocate online orders       1
for online retail orders                  1
Amazon                                    1
website fixed                             1
Name: count, dtype: int64[pyarrow]


StockCode : 21181


Description
PLEASE ONE PERSON METAL SIGN     1682
PLEASE ONE PERSON  METAL SIGN     181
<NA>                                3
adjustment                          2
missing                             1
on cargo order                      1
check                               1
dotcom                              1
Name: count, dtype: int64[pyarrow]


StockCode : 22734


Description
SET OF 6 RIBBONS VINTAGE CHRISTMAS     786
<NA>                                     5
amazon                                   3
Carton qnty was 216 not 144 as stat      1
amazon adjustment                        1
amendment                                1
amazon sales                             1
FOUND                                    1
Name: count, dtype: int64[pyarrow]

**Observation**
- certains `StockCode` possèdent plusieurs descriptions ;
- plusieurs variantes correspondent à des notes de correction ou de stock ;
- un même code peut aussi avoir de petites variations de libellé.

**Décision**
- utiliser `StockCode` comme identifiant principal de référence ;
- définir plus tard un libellé produit de référence à partir des descriptions valides.


### Quantités extrêmes

In [52]:
max_abs_quantity = df_consolidated["Quantity"].abs().max()

extreme_quantity = df_consolidated[
    df_consolidated["Quantity"].abs() == max_abs_quantity
].copy()

print("Quantité absolue maximale :", max_abs_quantity)

display(
    extreme_quantity[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country"
        ]
    ]
)


Quantité absolue maximale : 80995


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1043359,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446,United Kingdom
1043360,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446,United Kingdom


**Observation**
- quantité maximale absolue : 80 995
- même `StockCode`
- même client
- même prix
- une ligne positive puis une facture `C` négative 12 minutes plus tard

**Conclusion**
- ce cas illustre l'intérêt de conserver les annulations 
- un simple filtre sur les valeurs extrêmes supprimerait une information métier utile.


## 9. Synthèse de l'audit qualité

**1. Chevauchement des sources**
- 22 523 lignes et 1 088 factures présentes dans les deux feuilles.
- **Règle :** conserver une seule occurrence.

**2. Annulations et quantités négatives**
- 22 557 lignes avec `Quantity < 0`.
- 19 164 sont sur facture `C`.
- une facture `C` contient une quantité positive.
- **Règle :** `C` = annulation selon UCI ; signaler séparément les cas atypiques.

**3. Prix**
- 5 prix négatifs, 6 024 prix nuls et plusieurs valeurs très élevées.
- **Règle :** aucune suppression sur seuil ; classification selon le contexte.

**4. Customer ID**
- 235 287 lignes manquantes, soit 22,52 %.
- aucune facture ne mélange clients identifiés et non identifiés.
- **Règle :** aucune imputation ; séparation analyse commerciale / analyse client.

**5. Description**
- 4 275 lignes manquantes, toutes avec `Price = 0`.
- **Règle :** aucune imputation.

**6. Observations identiques**
- 11 812 lignes seraient retirées par `drop_duplicates()`.
- 1,13 % du volume mais 0,2868 % de la valeur brute.
- **Règle :** aucune suppression automatique.

**7. Préfixes de facture**
- trois formes observées : numérique, `C` et `A`.
- les 6 lignes `A` observées correspondent à `Adjust bad debt`.
- **Règle :** ne pas généraliser la signification de `A`.

**8. StockCode spéciaux**
- 5 992 lignes et 62 codes distincts ;
- les 62 codes ont été classés en 10 catégories analytiques ;
- aucun code ne reste non classé.
- **Règle :** conserver la classification dans Silver et l'utiliser pour définir le périmètre des KPI produits.

**9. Grain de la facture**
- un seul client et un seul pays par facture ;
- 83 factures possèdent deux horodatages, toutes sur une même journée ;
- écart médian : 1 minute ;
- écart maximal : 9 minutes.
- **Règle :** `order_datetime` = premier `InvoiceDate` observé pour la facture.

**10. Pays du client**
- 13 clients apparaissent dans deux pays.
- **Règle :** conserver `Country` au niveau transaction ou commande.

**11. Description produit**
- certains `StockCode` ont plusieurs descriptions.
- **Règle :** `StockCode` comme identifiant principal ; libellé de référence à définir.

**12. Quantités extrêmes**
- maximum absolu : 80 995 ;
- paire vente / annulation observée à 12 minutes d'écart.
- **Règle :** conserver les valeurs extrêmes et leur contexte.


## 10. Prochaine étape

L'audit qualité est terminé.

À faire ensuite
1. créer DuckDB et la couche Bronze ;
2. construire la couche Silver avec les règles issues de l'audit ;
3. définir les tables analytiques ;
4. commencer les analyses business et clients.


Le pipeline reproductible est développé dans `src/`.
